# Study area and national CROME subsetting

This notebook prepares the East Anglia study boundary and some area to explore.


In [ ]:
#Extract the national CROME 2022 archive to a dedicated runtime folder.
from pathlib import Path

CROME_ZIP = Path('/content/drive/MyDrive/Dissertation/Crop_Map_of_England_CROME_2022.geojson.zip')
CROME_EXTRACT_DIR = Path('/content/crome_2022_national')
CROME_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if not CROME_ZIP.exists():
    raise FileNotFoundError(f'Missing CROME archive: {CROME_ZIP}')

!unzip -o "{CROME_ZIP}" -d "{CROME_EXTRACT_DIR}"

#locate the national CROME GeoJSON
expected_name = 'crop_map_of_england_crome_2022.geojson'
crome_candidates = [
    path for path in CROME_EXTRACT_DIR.rglob('*.geojson')
    if path.name.lower() == expected_name
]

if len(crome_candidates) != 1:
    raise RuntimeError(
        f'Expected one {expected_name}, found {len(crome_candidates)} '
        f'in {CROME_EXTRACT_DIR}'
    )

CROME_GEOJSON = crome_candidates[0]
print('National CROME file:', CROME_GEOJSON)
print('File size (GB):', round(CROME_GEOJSON.stat().st_size / 1e9, 2))


In [ ]:
#Load and validate the administrative boundary layer.
import geopandas as gpd

BOUNDARY_PATH = Path(
    '/content/drive/MyDrive/uk_geo_data/uk_boundary/'
    'Counties_and_Unitary_Authorities_December_2024_Boundaries_UK_BGC_3152178837812104842.geojson'
)
NAME_FIELD = 'CTYUA24NM'

if not BOUNDARY_PATH.exists():
    raise FileNotFoundError(f'Missing boundary file: {BOUNDARY_PATH}')

gdf_boundaries = gpd.read_file(BOUNDARY_PATH)
required_areas = {
    'Hertfordshire', 'Essex', 'Southend-on-Sea', 'Thurrock',
    'Norfolk', 'Suffolk', 'Cambridgeshire', 'Peterborough',
    'Bedford', 'Central Bedfordshire', 'Luton',
}

if NAME_FIELD not in gdf_boundaries.columns:
    raise KeyError(f'Missing boundary name field: {NAME_FIELD}')
if gdf_boundaries.crs is None:
    raise ValueError('The boundary file has no CRS.')

missing_areas = required_areas.difference(gdf_boundaries[NAME_FIELD])
if missing_areas:
    raise ValueError(f'Missing administrative areas: {sorted(missing_areas)}')

gdf_boundaries = gdf_boundaries[gdf_boundaries[NAME_FIELD].isin(required_areas)].copy()
if gdf_boundaries.geometry.is_empty.any() or gdf_boundaries.geometry.isna().any():
    raise ValueError('The selected boundary layer contains empty geometries.')
if (~gdf_boundaries.geometry.is_valid).any():
    try:
        gdf_boundaries['geometry'] = gdf_boundaries.geometry.make_valid()
    except AttributeError:
        gdf_boundaries['geometry'] = gdf_boundaries.geometry.buffer(0)

selected_boundary_path = Path('/content/Selected_Regions_Boundaries.geojson')
gdf_boundaries.to_file(selected_boundary_path, driver='GeoJSON')

print('Boundary CRS:', gdf_boundaries.crs)
print('Selected administrative areas:', len(gdf_boundaries))
display(gdf_boundaries[[NAME_FIELD, 'geometry']])


In [ ]:
#create regional boundaries
regions_map = {
    'Hertfordshire': ['Hertfordshire'],
    'Essex': ['Essex', 'Southend-on-Sea', 'Thurrock'],
    'East_Anglia': ['Norfolk', 'Suffolk', 'Cambridgeshire', 'Peterborough'],
    'Bedfordshire': ['Bedford', 'Central Bedfordshire', 'Luton'],
}

for region_name, areas in regions_map.items():
    region_gdf = gdf_boundaries[gdf_boundaries[NAME_FIELD].isin(areas)]
    if region_gdf.empty:
        raise ValueError(f'No boundary found for {region_name}')

    output_file = Path(f'/content/{region_name}_Boundary.geojson')
    region_gdf.to_file(output_file, driver='GeoJSON')
    print(f'Saved {region_name}: {len(region_gdf)} administrative areas')


## CROME County Subset

Read national GeoJSON separately for each county bounding box to limit memory use.


In [ ]:
import pandas as pd
import shutil

COUNTY_GROUPS = {
    'Cambridgeshire': ['Cambridgeshire', 'Peterborough'],
    'Norfolk': ['Norfolk'],
    'Suffolk': ['Suffolk'],
}

OUTPUT_BASE = Path('/content/drive/MyDrive/Dissertation/crome/2022/crome_2022_selected')
OUTPUT_DIR = OUTPUT_BASE / 'east_anglia'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CREATE_ZIP = True #create ZIP because the output is large

# Read one feature to obtain the national CROME CRS without loading the full file.
crome_probe = gpd.read_file(CROME_GEOJSON, rows=1)
if crome_probe.crs is None:
    raise ValueError('The national CROME file has no CRS.')
crome_crs = crome_probe.crs
print('CROME CRS:', crome_crs)

summary_rows = []
for county_label, source_names in COUNTY_GROUPS.items():
    source_boundary = gdf_boundaries[
        gdf_boundaries[NAME_FIELD].isin(source_names)
    ].to_crs(crome_crs)

    try:
        county_geometry = source_boundary.geometry.union_all()
    except AttributeError:
        county_geometry = source_boundary.geometry.unary_union

    bbox = tuple(source_boundary.total_bounds)
    try:
        candidates = gpd.read_file(
            CROME_GEOJSON,
            bbox=bbox,
            engine='pyogrio',
            use_arrow=True,
        )
    except Exception:
        candidates = gpd.read_file(CROME_GEOJSON, bbox=bbox)

    if candidates.crs is None:
        candidates = candidates.set_crs(crome_crs)
    elif candidates.crs != crome_crs:
        candidates = candidates.to_crs(crome_crs)

    candidates = candidates[
        candidates.geometry.notna() & ~candidates.geometry.is_empty
    ].copy()
    representative_points = candidates.geometry.representative_point()
    county_cells = candidates[representative_points.within(county_geometry)].copy()
    county_cells['source_area'] = county_label

    output_path = OUTPUT_DIR / f'Crop_Map_of_England_2022_{county_label}.geojson'
    county_cells.to_file(output_path, driver='GeoJSON')

    summary_rows.append({
        'county_subset': county_label,
        'administrative_areas': '; '.join(source_names),
        'bbox_candidates': int(len(candidates)),
        'selected_crome_cells': int(len(county_cells)),
        'crs': str(county_cells.crs),
        'output_file': str(output_path),
    })
    print(f'{county_label}: {len(county_cells):,} CROME cells')

subset_summary = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / 'crome_2022_county_subset_summary.csv'
subset_summary.to_csv(summary_path, index=False)
display(subset_summary)

if CREATE_ZIP:
    archive = shutil.make_archive(
        str(OUTPUT_BASE),
        'zip',
        root_dir=OUTPUT_BASE.parent,
        base_dir=OUTPUT_BASE.name,
    )
    print('Created archive:', archive)

print('Saved subset summary:', summary_path)


In [ ]:
#merge other candidate region.
import pandas as pd

candidate_regions = ['Hertfordshire', 'Essex', 'Bedfordshire']
candidate_gdfs = [
    filtered_boundaries[filtered_boundaries[name_col].isin(regions_map[region])]
    for region in candidate_regions
]
candidate_gdfs = [gdf for gdf in candidate_gdfs if not gdf.empty]

if candidate_gdfs:
    merged_gdf = gpd.GeoDataFrame(
        pd.concat(candidate_gdfs, ignore_index=True),
        crs=candidate_gdfs[0].crs,
    )
    merged_output_path = '/content/Merged_Three_Regions_Boundary.geojson'
    merged_gdf.to_file(merged_output_path, driver='GeoJSON')
    print(f'Saved exploratory candidate boundary to {merged_output_path}')
